# Practice Lab: Multi-Agent with CrewAI & Agno (Lesson 11.5)

Module 11 · 8 exercises · CrewAI + Agno + Flows + YAML + token economics + frameworks + India + deploy

Exercises 1-5 run locally (pure Python simulations).


# Lesson 11.5: Multi-Agent with CrewAI & Agno

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python simulations). Ex 6-8 are design.


In [ ]:
import json
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): CrewAI Research Team


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class SimAgent:
    def __init__(self,role,goal,backstory,tools=None):
        self.role=role;self.goal=goal;self.backstory=backstory;self.tools=tools or []
        self.overhead=int((len(role.split())+len(goal.split())+len(backstory.split()))*1.3)
    def execute(self,task,context=""):
        tool_res=[t(task) for t in self.tools]
        tokens=self.overhead+int(len(task.split())*1.3)+int(len(context.split())*1.3)
        return {"output":f"[{self.role}] {task[:30]}... tools={len(self.tools)}","tokens":tokens,"tools":tool_res}

class SimCrew:
    def __init__(self,agents,tasks): self.agents=agents;self.tasks=tasks
    def kickoff(self):
        ctx="";trace=[];total=0
        for t in self.tasks:
            r=t["agent"].execute(t["desc"],ctx); ctx+=f"\n{r['output']}"; total+=r["tokens"]
            trace.append({"agent":t["agent"].role,"tokens":r["tokens"],"output":r["output"][:50]})
        return {"trace":trace,"total":total}

researcher=SimAgent("Researcher","Find course info","Expert edtech researcher",
    [lambda q:"GenAI: 14999 INR, 146hrs",lambda q:"GenAI: 15-40 LPA, 300% growth"])
analyst=SimAgent("Analyst","Analyze for career advice","Career counselor 10yr")
writer=SimAgent("Writer","Write student advisory","Student-friendly writer")

crew=SimCrew([researcher,analyst,writer],[
    {"desc":"Research GenAI course details and market","agent":researcher},
    {"desc":"Analyze research, ROI analysis","agent":analyst},
    {"desc":"Write 300-word student advisory","agent":writer}])

print("CrewAI Research Team:")
r=crew.kickoff()
for t in r["trace"]: print(f"  [{t['agent']}] {t['tokens']} tokens: {t['output']}")
print(f"\n  Total: {r['total']} tokens (~56% overhead per agent)")
print(f"  output_pydantic: {json.dumps({'title':'GenAI Advisory','rec':'Enroll','roi_months':6})}")


</details>


---
## Exercise 2 (Easy): Agno Team Routing


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class SimAgno:
    def __init__(self,name,kws,cost): self.name=name;self.kws=kws;self.cost=cost
    def handle(self,q): return {"agent":self.name,"cost":self.cost}

class SimTeam:
    def __init__(self,agents): self.agents=agents
    def route(self,q):
        scores={a.name:sum(1 for k in a.kws if k in q.lower()) for a in self.agents}
        best=max(scores,key=scores.get)
        return next(a for a in self.agents if a.name==best).handle(q)
    def collaborate(self,q): return {"agents":len(self.agents),"cost":sum(a.cost for a in self.agents)}

team=SimTeam([
    SimAgno("Billing",["refund","payment","price","cost","emi"],0.001),
    SimAgno("Technical",["error","bug","install","setup","code"],0.01),
    SimAgno("Courses",["course","enroll","genai","python","schedule"],0.003)])

print("Agno Team Routing:")
queries=[("I want a refund","billing"),("Code won't run","technical"),("GenAI courses?","courses"),
         ("Python cost?","billing"),("Install failing","technical"),("Next class?","courses")]
route_cost=0
for q,_ in queries:
    r=team.route(q); route_cost+=r["cost"]
    print(f"  '{q[:22]}' -> {r['agent']} (${r['cost']:.3f})")

collab_cost=team.collaborate("")["cost"]*len(queries)
print(f"\n  Route: ${route_cost:.3f} vs Collaborate: ${collab_cost:.3f} = {(1-route_cost/collab_cost)*100:.0f}% savings")


</details>


---
## Exercise 3 (Medium): CrewAI Flows Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

class SimFlow:
    def __init__(self,threshold=0.7): self.threshold=threshold;self.attempts=0;self.scores=[]
    def run(self,topic,max_retries=3):
        for _ in range(max_retries+1):
            self.attempts+=1
            score=min(0.55+0.15*self.attempts+np.random.uniform(-0.1,0.1),1.0)
            self.scores.append(round(score,2))
            if score>=self.threshold: return {"status":"proceed","score":score}
        return {"status":"best_effort","score":self.scores[-1]}

print("CrewAI Flows Pipeline:")
flow=SimFlow(0.7)
r=flow.run("GenAI advisory")
print(f"  Attempts: {flow.attempts}, Scores: {flow.scores}")
print(f"  Final: {r['status']} (score={r['score']:.2f})")
print(f"\n  Flow: @start -> research -> @listen(evaluate) -> @router -> retry/proceed")
print(f"  Pipelines DEPRECATED. Use Flows. DocuSign: 14x less code than LangGraph")


</details>


---
## Exercise 4 (Medium): YAML-Driven Crew


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

agents_yaml={"researcher":{"role":"Course Researcher","goal":"Find {topic} info"},
    "analyst":{"role":"Career Analyst","goal":"Analyze {topic} data"},
    "writer":{"role":"Report Writer","goal":"Write {topic} advisory"}}
tasks_yaml={"research":{"agent":"researcher","context":[]},
    "analysis":{"agent":"analyst","context":["research"]},
    "report":{"agent":"writer","context":["research","analysis"]}}

print("YAML-Driven Crew:")
print("  agents.yaml:")
for n,c in agents_yaml.items(): print(f"    {n}: {c['role']} -> {c['goal']}")
print("  tasks.yaml:")
for n,c in tasks_yaml.items(): print(f"    {n}: agent={c['agent']}, deps={c['context']}")
topic="GenAI"
print(f"\n  {{variable}} interpolation (topic={topic}):")
for n,c in agents_yaml.items(): print(f"    {n}.goal: {c['goal'].replace('{topic}',topic)}")
print(f"\n  YAML: ~30 lines, editable by anyone | Code: ~50 lines, developers only")


</details>


---
## Exercise 5 (Medium): Token Economics Audit


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Token Economics Audit:")
single_in,single_out=500,300; single_total=800
crew_in=500+150 + 500+150+200 + 500+150+450; crew_out=200+250+300; crew_total=crew_in+crew_out

single_cost=(single_in*3+single_out*15)/1e6
crew_cost=(crew_in*3+crew_out*15)/1e6

print(f"  {'':16} {'Single':>10} {'3-Agent':>10} {'Ratio':>8}")
print(f"  {'Input':16} {single_in:>10} {crew_in:>10} {crew_in/single_in:>7.1f}x")
print(f"  {'Output':16} {single_out:>10} {crew_out:>10} {crew_out/single_out:>7.1f}x")
print(f"  {'Total':16} {single_total:>10} {crew_total:>10} {crew_total/single_total:>7.1f}x")
print(f"  {'Cost(Sonnet)':16} ${single_cost:>9.4f} ${crew_cost:>9.4f} {crew_cost/single_cost:>7.1f}x")

reduced=crew_cost*(1-0.90)*(1-0.60)*(1-0.30)
print(f"\n  After caching+routing+pydantic: ${reduced:.6f} ({(1-reduced/crew_cost)*100:.0f}% reduction)")
print(f"  70% of use cases work with single agent")
print(f"  Multi-agent justified: compliance isolation, critic patterns, complex planning")


</details>


---
## Exercise 6 (Challenge): Cross-Framework Comparison
Design/architecture. See practice-lab-11_5.html.


In [ ]:
# Exercise 6: Cross-Framework Comparison
pass


<details><summary>Solution</summary>


In [ ]:
print('5 Frameworks: CrewAI(46K,fast) Agno(39K,perf) LangGraph(25K,state) OpenAI(21K,minimal) ADK(16K,GCP)')
print('Decision: durable state->LangGraph, fast proto->CrewAI, prod runtime->Agno')
print('MCP for tools + A2A for agents = framework-agnostic interop')
print('Trajectory: prototype CrewAI, ship LangGraph')


</details>


---
## Exercise 7 (Challenge): Multilingual Agent Team
Design/architecture. See practice-lab-11_5.html.


In [ ]:
# Exercise 7: Multilingual Agent Team
pass


<details><summary>Solution</summary>


In [ ]:
print('Multilingual Team: Hindi->Sarvam-105b(FREE) + English->GPT-4o($3/MTok)')
print('1000 queries (60% Hindi): GPT-4o=$3.00 vs Routed=$1.20 = 60% savings')
print('India Stack: Aadhaar $0.15, UPI free<2K, DigiLocker free, ONDC open')
print('DPDP consent gate MANDATORY before India Stack calls')


</details>


---
## Exercise 8 (Challenge): Production Deploy
Design/architecture. See practice-lab-11_5.html.


In [ ]:
# Exercise 8: Production Deploy
pass


<details><summary>Solution</summary>


In [ ]:
print('Deploy: crewai create -> crewai run -> docker build -> gcloud run deploy')
print('Observability: AgentOps($0-36) LangSmith($0-500) OpenLIT(free) Datadog(enterprise)')
print('Model routing: 90% Flash + 10% Sonnet = 87% savings')
print('Alerts: cost>$50/day, p95>30s, errors>5%, tokens>50K/task')


</details>
